In [1]:
level_depth=0

In [2]:
import os
import argparse
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
import plotly.io as pio
from pathlib import Path
from bertopic import BERTopic
import pandas as pd
import plotly.io as pio
from bertopic import BERTopic
from pathlib import Path
import pandas as pd
import plotly.express as px
import pandas as pd
import numpy as np
import joblib

# Headless-safe Plotly renderer
pio.renderers.default = "notebook_connected" if (os.environ.get("DISPLAY") or os.environ.get("MPLBACKEND")) else "png"

print(f"[INFO] level_depth={level_depth}")

/home/students/s328743/.conda/envs/bertopic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] level_depth=0


In [3]:
#paths
path_df_sampled = f"../../results/levels/level_{level_depth}/grid_search/df_sampled_level_{level_depth}.csv"
best_dir = f"../../results/levels/level_{level_depth}/best"
best_model_path = os.path.join(best_dir, "best_model.pkl")
df_spam_messages_path = f"../../results/levels/level_{level_depth}/preProcessing/preprocessed_spam_messages_level_{level_depth}.tsv.gz"
path_summary_filtered = f"../../results/levels/level_{level_depth}/percentage_of_politics_msgs/summary_pol_sorted_filtered_spam.csv.gz"

# best_model_path deve puntare a un file .pkl
if not os.path.isfile(best_model_path):
    raise FileNotFoundError(f"Best model file not found: {best_model_path}")

print(f"[LOAD] joblib.load -> {best_model_path}")
topic_model: BERTopic = joblib.load(best_model_path)


# Attach the assigned topic to each message
df_sampled = pd.read_csv(path_df_sampled)
df_sampled["topic"] = topic_model.topics_

print(f"number of initial seeds {len(df_sampled)}\n")
print(f"df_sampled:\n {df_sampled.head()}")

#DF_SPAM_MESSAGES
df_spam_messages = pd.read_csv(df_spam_messages_path, sep='\t', compression='gzip')
print(f"df_spam_messages:\n {df_spam_messages.head()}")

df_total_spam = (
    df_spam_messages
    .groupby("channel_id")["count_spam"]
    .sum()
    .reset_index(name="spam_msgs")
)
print("\n----\n")
print(f"df_total_spam:\n {df_total_spam}")

#summary_pol_sorted_filtered_spam
summary_pol_sorted_filtered_spam  = pd.read_csv(path_summary_filtered, compression="gzip")
print(f"summary_pol_sorted_filtered_spam: \n {summary_pol_sorted_filtered_spam.head()}")





[LOAD] joblib.load -> ../../results/levels/level_0/best/best_model.pkl
number of initial seeds 30054

df_sampled:
                                                 text   timestamp  \
0  💊 DEBATE [36.3K/1.9%] $DEBATE 🔼\n🌐 Solana @ Ra...  1725920360   
1  We will make America greater than ever before ...  1702146126   
2  JD Vance: Whatever the number is, it's way too...  1724594968   
3  Japanese Scientists Find Indisputable Evidence...  1702808169   
4  Pamela's friend told me that she is on holiday...  1706170568   

                                   text_preprocessed language  \
0  debate debate solana raydium xvol mint tgxweb ...       en   
1      america greater support join share join storm       en   
2  vance number high millions illegals come kamal...       en   
3  japanese scientists find indisputable evidence...       en   
4  pamelafriend told holiday queenstown wait post...       en   

           channel_id  topic  
0  channel_2065916066      0  
1  channel_1906835275  

In [4]:
#numer of unique channels
print("number of unique channels")
print(summary_pol_sorted_filtered_spam['channel_id'].nunique())

# Count the number of channels in each bin
distribution = summary_pol_sorted_filtered_spam["pol_ratio_without_spam_range"].value_counts().sort_index()

# Print the result
print("Distribution of channels by political content ratio (>30%):\n")
for label, count in distribution.items():
    print(f"{label}: {count} channels")


distribution_plot = summary_pol_sorted_filtered_spam["pol_ratio_without_spam_range"].value_counts().sort_index()

# Forza ordinamento corretto
ordered_labels = ["0%", "10%", "20%", "30%", "40%", "50%", "60%", "70%", "80%", "90%"]
distribution_plot = distribution_plot.reindex(ordered_labels).fillna(0)

# Converte in DataFrame
distribution_df = distribution_plot.reset_index()
distribution_df.columns = ["Political Content Ratio (bins)", "Number of Channels"]

# Crea il grafico
fig = px.bar(
    distribution_df,
    x="Political Content Ratio (bins)",
    y="Number of Channels",
    title="Distribution of Channels by Political Content Ratio"
)

# Mostra il grafico (solo se supportato)
fig.show()





number of unique channels
24
Distribution of channels by political content ratio (>30%):

0%: 6 channels
10%: 1 channels
20%: 4 channels
30%: 2 channels
40%: 6 channels
50%: 3 channels
60%: 2 channels
